In [1]:
import pandas as pd

In [2]:
file_path = '../data/test/07-03-2025-PO.csv'

In [3]:
def load_df(file_path):
    df = pd.read_csv(file_path)
    df = df.dropna(axis=1)
    df = df.drop_duplicates(subset='CODE')
    if df['CODE'].dtype in ['float', 'int', 'int64']:
        df['CODE'] = df['CODE'].astype(int)
        df['CODE'] = df['CODE'].astype(str)
    return df

In [4]:
data = load_df(file_path)

In [5]:
print(data.columns)
print(data.shape)

Index(['CODE', 'SALE'], dtype='object')
(23, 2)


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23 entries, 0 to 22
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   CODE    23 non-null     object 
 1   SALE    23 non-null     float64
dtypes: float64(1), object(1)
memory usage: 496.0+ bytes


In [7]:
data['SALE'].sum()

466742.17

In [8]:
master_data = pd.read_csv('../data/master/master_gps_data.csv')

In [9]:
print(master_data.columns)
print(master_data.shape)

Index(['CODE', 'LOCATION', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'BRAND',
       'DISTRICT'],
      dtype='object')
(650, 7)


In [10]:
"""Add LOCATION,ADDRESS,LATITUDE,LONGITUDE,BRAND columns from master_data to data for relevant CODE"""

data_enriched = pd.merge(
    data,
    master_data[['CODE', 'LOCATION', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'BRAND', 'DISTRICT']],
    on='CODE',
    how='left'
)

# Show result
print(data_enriched.columns)
print(data_enriched.shape)

Index(['CODE', 'SALE', 'LOCATION', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'BRAND',
       'DISTRICT'],
      dtype='object')
(23, 8)


In [ ]:
import pandas as pd

MASTER_PATH = '../data/master/master_gps_data.csv'

def get_enriched_data(file_path: str) -> pd.DataFrame:
    # Load the primary data
    df = pd.read_csv(file_path)
    df = df.dropna(axis=1)
    df = df.drop_duplicates(subset='CODE')
    
    if df['CODE'].dtype in ['float', 'int', 'int64']:
        df['CODE'] = df['CODE'].astype(int)
        df['CODE'] = df['CODE'].astype(str)

    # Load the master data
    master_df = pd.read_csv(MASTER_PATH)

    # Merge with master data on CODE
    enriched_df = pd.merge(
        df,
        master_df[['CODE', 'LOCATION', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'BRAND', 'DISTRICT']],
        on='CODE',
        how='left'
    )

    return enriched_df

In [2]:
day = '03-03-2025'

In [3]:
# Example usage
file_path = f'../data/test/{day}-PO.csv'
save_path = f'../data/test/orders/{day}-PO.csv'

final_df = get_enriched_data(file_path)
final_df['DATE']= day

final_df.shape

(457, 9)

In [4]:
final_df.to_csv(save_path, index=False)

In [1]:
import pandas as pd

In [59]:
day = "18-07-2025"
po_file_path = f"../data/test/po-volume/{day}-PO.csv"

In [60]:
# def load_df(file_path):
#     df = pd.read_csv(file_path)
#     df = df.dropna(axis=1)
#     df = df.drop_duplicates(subset='CODE')
#     if df['CODE'].dtype in ['float', 'int', 'int64']:
#         df['CODE'] = df['CODE'].astype(int)
#         df['CODE'] = df['CODE'].astype(str)
#     return df

# df = load_df(po_file_path)

In [61]:
MASTER_PATH = '../data/master/master_gps.csv'

def get_enriched_data(file_path: str) -> pd.DataFrame:
    # Load the primary data
    df = pd.read_csv(file_path)
    df = df.dropna(axis=1)
    df = df.drop_duplicates(subset='CODE')
    df['DATE'] = day
    
    if df['CODE'].dtype in ['float', 'int', 'int64']:
        df['CODE'] = df['CODE'].astype(int)
        df['CODE'] = df['CODE'].astype(str)

    # Load the master data
    master_df = pd.read_csv(MASTER_PATH)

    # Merge with master data on CODE
    enriched_df = pd.merge(
        df,
        master_df[['CODE', 'LOCATION', 'ADDRESS', 'LATITUDE', 'LONGITUDE', 'BRAND', 'DISTRICT']],
        on='CODE',
        how='left'
    )

    return enriched_df

In [62]:
po_df = get_enriched_data(po_file_path)

In [63]:
save_path = f"../data/test/orders-vol/{day}-PO.csv"
po_df.to_csv(save_path, index=False)

In [13]:
import pandas as pd

In [14]:
cargils_master_df = pd.read_csv('../data/test/cargils_master.csv')

In [60]:
day = "17-07-2025"
po_df = pd.read_csv(f"../data/test/orders-vol/{day}-PO.csv")

In [61]:
marge_df = pd.merge(
    po_df,
    cargils_master_df[['CODE','LOCATION']],
    on='CODE',
    how='left', 
    suffixes=('', '_master')
)

marge_df.columns

Index(['CODE', 'SALE', 'VOLUME', 'DATE', 'LOCATION', 'ADDRESS', 'LATITUDE',
       'LONGITUDE', 'BRAND', 'DISTRICT', 'LOCATION_master'],
      dtype='object')

In [62]:
po_df['LOCATION'] = po_df['LOCATION'].fillna(marge_df['LOCATION_master'])

In [63]:
def add_address(value:str) -> str:
    if pd.isna(value):
        return 
    
    address = ' '.join(value.split(' ')[1:])
    
    return address

In [64]:
add_address(po_df['LOCATION'][0])

'NEGOMBO'

In [65]:
po_df['ADDRESS'] = po_df['LOCATION'].apply(add_address)

In [66]:
po_df['BRAND'] = 'Cargills'

In [67]:
po_df['LOCATION'].isna().sum()

0

In [15]:
import pandas as pd
import requests
from dotenv import load_dotenv
import os

_ = load_dotenv()

gcp_key = os.environ['GOOGLE_MAPS_API_KEY']

In [16]:
def get_gcp_gps(address):
    """
    Geocodes an address and returns (latitude, longitude, district) as a tuple.
    
    Args:
        address (str): The address to geocode.
    
    Returns:
        pd.Series: Series with latitude, longitude, and district.
    """
    try:
        address += ", Sri lanka"
        # Construct the URL
        url = f"https://maps.googleapis.com/maps/api/geocode/json?address={address}&key={gcp_key}"

        # Send the request
        response = requests.get(url)

        # Parse the JSON response
        if response.status_code == 200:
            data = response.json()
            if data['status'] == 'OK' and data['results']:
                location = data['results'][0]['geometry']['location']
                latitude = location['lat']
                longitude = location['lng']
                district = None
                for component in data['results'][0]['address_components']:
                    if 'administrative_area_level_2' in component['types']:
                        district = component['long_name']
                        break
                # Fallback to administrative_area_level_1 if level_2 is not found
                if not district:
                    for component in data['results'][0]['address_components']:
                        if 'administrative_area_level_1' in component['types']:
                            district = component['long_name']
                            break
                # If still no district, use a default value
                district = district if district else 'Unknown'
                print(f"Address: {address}, Latitude: {latitude}, Longitude: {longitude}, District: {district}")
                return pd.Series([latitude, longitude, district])
            else:
                print(f"Geocoding error for address '{address}': {data.get('status', 'Unknown')}")
                return pd.Series([None, None, 'Unknown'])
        else:
            print(f"HTTP Error for address '{address}': {response.status_code}")
            return pd.Series([None, None, 'Unknown'])
    except Exception as e:
        print(f"Exception for address '{address}': {e}")
        return pd.Series([None, None, 'Unknown'])

In [17]:
get_gcp_gps("Mirissa station, udupila")

Address: Mirissa station, udupila, Sri lanka, Latitude: 5.956849999999999, Longitude: 80.47337999999999, District: Matara


0     5.95685
1    80.47338
2      Matara
dtype: object

In [18]:
def get_coords_and_district(addr):
    return get_gcp_gps(addr)

In [72]:
po_df[['LATITUDE','LONGITUDE', 'DISTRICT']] = po_df['ADDRESS'].apply(get_coords_and_district)

Address: NEGOMBO, Sri lanka, Latitude: 7.205520799999999, Longitude: 79.8512562, District: Gampaha
Address: JAELA, Sri lanka, Latitude: 7.0667984, Longitude: 79.90409319999999, District: Gampaha
Address: CHILAW, Sri lanka, Latitude: 7.577715500000001, Longitude: 79.7943865, District: Puttalam
Address: KADAWATHA, Sri lanka, Latitude: 7.004632399999999, Longitude: 79.954155, District: Gampaha
Address: WENNAPPUWA, Sri lanka, Latitude: 7.341191200000001, Longitude: 79.84200129999999, District: Puttalam
Address: PELIYAGODA, Sri lanka, Latitude: 6.9574476, Longitude: 79.88954869999999, District: Gampaha
Address: KANDANA, Sri lanka, Latitude: 7.0477897, Longitude: 79.8970348, District: Gampaha
Address: SEEDUWA, Sri lanka, Latitude: 7.1247085, Longitude: 79.87500279999999, District: Gampaha
Address: NITTAMBUWA, Sri lanka, Latitude: 7.142363899999999, Longitude: 80.1037721, District: Gampaha
Address: BIA KATUNAYAKE, Sri lanka, Latitude: 7.1801543, Longitude: 79.8842495, District: Gampaha
Addres

In [57]:
po_df

,CODE,SALE,VOLUME,DATE,LOCATION,ADDRESS,LATITUDE,LONGITUDE,BRAND,DISTRICT
0,1068,20546.99,127387.00,18-07-2025,FC NEGOMBO,NEGOMBO,7.205521,79.851256,Cargills,Gampaha
1,1087,27580.15,155269.90,18-07-2025,FC JAELA,JAELA,7.066798,79.904093,Cargills,Gampaha
2,1102,28165.50,168753.80,18-07-2025,FC CHILAW,CHILAW,7.577716,79.794387,Cargills,Puttalam
3,1112,12509.21,110755.90,18-07-2025,FC KADAWATHA,KADAWATHA,7.004632,79.954155,Cargills,Gampaha
4,1131,11338.03,126951.80,18-07-2025,FC WENNAPPUWA,WENNAPPUWA,7.341191,79.842001,Cargills,Puttalam
...,...,...,...,...,...,...,...,...,...,...
65,1855,11733.82,94979.88,18-07-2025,EX MIRISWATTA,MIRISWATTA,7.072739,80.015763,Cargills,Gampaha
66,1856,11138.11,73423.92,18-07-2025,EX KIMBULAPITIYA,KIMBULAPITIYA,7.204241,79.894081,Cargills,Gampaha
67,1857,21002.39,120195.80,18-07-2025,EX WILIMBULA,WILIMBULA,7.873054,80.771797,Cargills,Unknown
68,1861,35289.08,207815.20,18-07-2025,EX KATANA,KATANA,7.248028,79.899366,Cargills,Gampaha


In [58]:
get_gcp_gps("WILIMBULA, sri lanka")

Address: WILIMBULA, sri lanka, Sri lanka, Latitude: 7.873053999999999, Longitude: 80.77179699999999, District: Unknown


0     7.873054
1    80.771797
2      Unknown
dtype: object

In [73]:
save_path = f"../data/test/orders-vol/{day}-PO.csv"
po_df.to_csv(save_path, index=False)

In [1]:
import pandas as pd

In [3]:
day = '14-07-2025'
po_df = pd.read_csv(f'../data/test/orders-vol/{day}-PO.csv')

In [4]:
po_df

,CODE,SALE,VOLUME,DATE,LOCATION,ADDRESS,LATITUDE,LONGITUDE,BRAND,DISTRICT
0,2,16348.52,64255.92,14-07-2025,NaN,NaN,NaN,NaN,NaN,NaN
1,3,47333.79,216338.60,14-07-2025,Wattala SC,"Wattala, Sri Lanka",6.990668,79.893171,Arpico,Gampaha
2,4,2453.86,10560.00,14-07-2025,Borelasgamuwa SS,"Borelasgamuwa, Sri Lanka",6.840989,79.901719,Arpico,Colombo
3,5,36787.44,165178.80,14-07-2025,Hyde Park SC,"Hyde, Sri Lanka",6.917587,79.858519,Arpico,Colombo
4,10,16811.30,76639.92,14-07-2025,Nawinna SC,"Nawinna, Sri Lanka",6.853331,79.915072,Arpico,Colombo
...,...,...,...,...,...,...,...,...,...,...
476,Rathmalana,99402.40,487861.60,14-07-2025,NaN,NaN,NaN,NaN,NaN,NaN
477,Seeduwa,54085.20,341975.50,14-07-2025,NaN,NaN,NaN,NaN,NaN,NaN
478,Wattala,77921.44,460975.20,14-07-2025,NaN,NaN,NaN,NaN,NaN,NaN
479,Wellawatta,90996.68,746953.00,14-07-2025,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
laugf_df = po_df.iloc[444:]

In [13]:
laugf_df['BRAND'] = 'Laugf'
laugf_df['DATE'] = '14-07-2025'
laugf_df['LOCATION'] = laugf_df['CODE']
laugf_df['ADDRESS'] = laugf_df['CODE']

C:\Users\HP\AppData\Local\Temp\ipykernel_18436\2650518573.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laugf_df['BRAND'] = 'Laugf'
C:\Users\HP\AppData\Local\Temp\ipykernel_18436\2650518573.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laugf_df['DATE'] = '14-07-2025'
C:\Users\HP\AppData\Local\Temp\ipykernel_18436\2650518573.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 

In [19]:
laugf_df[['LATITUDE', 'LONGITUDE', 'DISTRICT']] = laugf_df['ADDRESS'].apply(get_coords_and_district)

Address: Baseline, Sri lanka, Latitude: 6.9350919, Longitude: 79.8783278, District: Western Province
Address: Biyagama, Sri lanka, Latitude: 6.9462153, Longitude: 79.9892034, District: Gampaha
Address: Boralesgamuwa, Sri lanka, Latitude: 6.8409891, Longitude: 79.90171869999999, District: Colombo
Address: Chilaw, Sri lanka, Latitude: 7.577715500000001, Longitude: 79.7943865, District: Puttalam
Address: Delgoda, Sri lanka, Latitude: 6.987854, Longitude: 80.0157633, District: Gampaha
Address: Ederamulla, Sri lanka, Latitude: 6.997783699999999, Longitude: 79.91898139999999, District: Gampaha
Address: Hanwella, Sri lanka, Latitude: 6.8978344, Longitude: 80.0814292, District: Colombo
Address: Havelock Town, Sri lanka, Latitude: 6.886651899999999, Longitude: 79.8646835, District: Colombo
Address: Hokandara, Sri lanka, Latitude: 6.8743192, Longitude: 79.96962769999999, District: Colombo
Address: Jubilee Post, Sri lanka, Latitude: 6.875126799999999, Longitude: 79.90143379999999, District: Colom

C:\Users\HP\AppData\Local\Temp\ipykernel_18436\1701404788.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  laugf_df[['LATITUDE', 'LONGITUDE', 'DISTRICT']] = laugf_df['ADDRESS'].apply(get_coords_and_district)


In [29]:
process_df = po_df.combine_first(laugf_df)

In [30]:
process_df

,CODE,SALE,VOLUME,DATE,LOCATION,ADDRESS,LATITUDE,LONGITUDE,BRAND,DISTRICT
0,2,16348.52,64255.92,14-07-2025,NaN,NaN,NaN,NaN,NaN,NaN
1,3,47333.79,216338.60,14-07-2025,Wattala SC,"Wattala, Sri Lanka",6.990668,79.893171,Arpico,Gampaha
2,4,2453.86,10560.00,14-07-2025,Borelasgamuwa SS,"Borelasgamuwa, Sri Lanka",6.840989,79.901719,Arpico,Colombo
3,5,36787.44,165178.80,14-07-2025,Hyde Park SC,"Hyde, Sri Lanka",6.917587,79.858519,Arpico,Colombo
4,10,16811.30,76639.92,14-07-2025,Nawinna SC,"Nawinna, Sri Lanka",6.853331,79.915072,Arpico,Colombo
...,...,...,...,...,...,...,...,...,...,...
476,Rathmalana,99402.40,487861.60,14-07-2025,Rathmalana,Rathmalana,6.819545,79.880083,Laugf,Colombo
477,Seeduwa,54085.20,341975.50,14-07-2025,Seeduwa,Seeduwa,7.124708,79.875003,Laugf,Gampaha
478,Wattala,77921.44,460975.20,14-07-2025,Wattala,Wattala,6.990668,79.893171,Laugf,Gampaha
479,Wellawatta,90996.68,746953.00,14-07-2025,Wellawatta,Wellawatta,6.875531,79.860998,Laugf,Colombo


In [31]:
process_df.to_csv(f'../data/test/orders-vol/{day}-PO-2.csv', index=False)

In [32]:
laugf_df.to_csv(f'../data/test/orders-vol/{day}-PO-3.csv', index=False)